In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import kagglehub
import os
from tqdm import tqdm

df = pd.read_csv(path+'/Q1_data.csv')

In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# Task 5: Write your code here:
def check_target_distribution(df, target_column):
  df[target_column].hist(bins=30, edgecolor='black')

  plt.title(f"Target Distribution ({target_column})")
  plt.xlabel(target_column)
  plt.ylabel("Frequency")
  plt.grid(False)

  plt.show()

check_target_distribution(df, "Delivery_Time")

In [ ]:
# Task 1: Write your code here:
df = df.drop(["Order_ID"], axis = 1)
df.head()



In [ ]:
# Task 2: Write your code here:
def check_missing_values(df):
  missing_values = df.isnull().sum()
  print("Missing Values per Column:")
  print(missing_values[missing_values > 0])
  if missing_values.any():
    print("\nHandle Missing Values as needed.")
  else:
    print("\nNo Missing Values Found.")
#checking for missing values
check_missing_values(df)

#handling catgorical missing values
for col in ["Weather", "Traffic_Level", "Time_of_Day"]:
  df[col] = df[col].fillna(df[col].mode()[0])

#handling numerical columns
for col in ["Courier_Experience_yrs", "Delivery_Time"]:
  df[col] = df[col].fillna(df[col].mean())

print("After handling missing data: ")

check_missing_values(df)

In [ ]:
# Task 3: Write your code here:
def check_duplicates(df):
  duplicates = df.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")
check_duplicates(df)

In [ ]:
# Task 4: Write your code here:
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import LabelEncoder

one_hot_encoder = OneHotEncoder(sparse_output=False)
label_encoder = LabelEncoder()

df["Traffic_Level"] = label_encoder.fit_transform(df["Traffic_Level"])

df["Weather"] = label_encoder.fit_transform(df["Weather"])

df["Vehicle_Type"] = label_encoder.fit_transform(df["Vehicle_Type"])

df["Time_of_Day"] = label_encoder.fit_transform(df["Time_of_Day"])

df.head()






In [ ]:
# Task 5: Write your code here:
from sklearn.preprocessing import MinMaxScaler

features = df.columns.drop("Delivery_Time")  # DON'T SCALE THE TARGET

scaler = MinMaxScaler()
df[features] = scaler.fit_transform(df[features])
df.head()

In [ ]:
# Task 6: Write your code here:

#Since it is a countinuos variable no need for target imbalance aka we are not doing classification

In [ ]:
# Task 1: Write your code here:

X = df.drop("Delivery_Time", axis=1).astype(float)
y = df['Delivery_Time'].astype(float)


In [ ]:
# Task 2,3,4,5: Write your code here:
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error
from sklearn.ensemble import RandomForestRegressor
n_splits = 5  # K=5 Folds

# 5-Fold Cross-Validation, shuffled
kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

maes = []

for fold_idx, (train_index, test_index) in enumerate(kf.split(X)):
  print(f"\nFold {fold_idx + 1}/{n_splits}")

  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]
  rf = RandomForestRegressor(n_estimators=200)
  rf.fit(X_train, y_train)
  y_pred = rf.predict(X_test)
  mae = mean_absolute_error(y_test, y_pred)
  print("The mae for this fold is ", mae)
  maes.append(mae)
avg_score = 0
for mae in maes:
  avg_score+=mae
print("The average score is ", avg_score/n_splits)


In [ ]:
# Task 1: Write your code here:
importance = pd.DataFrame({
    'feature': ["Courier_Experience_yrs", "Weather", "Preparation_Time_min","Vehicle_Type","Time_of_Day"],
    'importance': rf.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(importance['feature'], importance['importance'], color='purple')
plt.xlabel('Importance')
plt.title('Feature Importance')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
# Task 2: Write your code here:

plt.hist(y_pred,bins=30, edgecolor='black')

plt.title(f"Target Distribution ({"Predicted"})")
plt.xlabel("Predicted")
plt.ylabel("Frequency")
plt.grid(False)

plt.show()


In [ ]:
# Task Bonus: Write your code here: